# CAP4D Static Avatar for Unity (Colab)
This notebook runs tracking + generation + avatar fitting and exports a static Unity-friendly 3DGS `.ply`.

In [1]:
# Top-level settings
QUALITY = "max"   # "balanced" | "max" | "debug"
MAX_N_REF = 100           # reduce for speed, increase for quality
TIMESTEP = 0             # static FLAME timestep to bake
INPUT_VIDEO_PATH = "/content/my_head_video.mp4"
OUTPUT_PATH = "/content/cap4d/examples/output/custom_static"
REPO_URL = "https://github.com/vikram-menon/cap4d.git"  # set to your fork if needed
REPO_REF = "colab"  # branch/tag/commit containing static export scripts

In [2]:
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-402255ba-8c9f-86bc-5fbe-8cf4c6dcf36a)


In [3]:
import subprocess
%cd /content
!rm -rf cap4d
subprocess.run(["git", "clone", REPO_URL, "cap4d"], check=True)
%cd /content/cap4d
subprocess.run(["git", "checkout", REPO_REF], check=True)

/content
/content/cap4d


CompletedProcess(args=['git', 'checkout', 'colab'], returncode=0)

In [4]:
import os
os.environ["CAP4D_PATH"] = "/content/cap4d"
os.environ["PIXEL3DMM_PATH"] = "/content/pixel3dmm"
os.environ["PYTHONPATH"] = f"/content/cap4d:{os.environ.get('PYTHONPATH', '')}"
print('CAP4D_PATH=', os.environ['CAP4D_PATH'])
print('PIXEL3DMM_PATH=', os.environ['PIXEL3DMM_PATH'])

CAP4D_PATH= /content/cap4d
PIXEL3DMM_PATH= /content/pixel3dmm


In [6]:
%%bash
set -e
cd /content/cap4d
pip install -r requirements.txt
export FORCE_CUDA=1
pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"
apt-get update
apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 23.2 MB/s eta 0:00:00
  Created wheel for chum

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-50rgejzz
  Running command git checkout -q 75ebeeaea0908c5527e7b1e305fbc7681382db47
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os
import getpass
os.environ['FLAME_USERNAME'] = input('FLAME username: ')
os.environ['FLAME_PWD'] = getpass.getpass('FLAME password: ')

FLAME username: pigeenspam1@gmail.com
FLAME password: ··········


In [7]:
%%bash
set -e
cd /content/cap4d
bash scripts/download_flame.sh
bash scripts/download_mmdm_weights.sh
bash scripts/install_pixel3dmm.sh


If you do not have an account you can register at https://flame.is.tue.mpg.de/ following the installation instruction.

Archive:  FLAME2023.zip
  inflating: ./FLAME2023/Readme.pdf  
  inflating: ./FLAME2023/flame2023.pkl  
  inflating: ./FLAME2023/flame2023_no_jaw.pkl  

Installation has finished. If there were any error messages, follow README to download and unzip FLAME2023 manually!
  Cloning https://github.com/edavalosanaya/L2CS-Net.git (to revision main) to /tmp/pip-req-build-ws9fn06z
  Resolved https://github.com/edavalosanaya/L2CS-Net.git to commit 4a0f978d5b4c426a7d37022d8c927d6ea031dcb6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/elliottzheng/face-detection to /tmp/p

--2026-03-11 23:44:50--  https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2023.zip&resume=1
Resolving download.is.tue.mpg.de (download.is.tue.mpg.de)... 192.124.27.139
Connecting to download.is.tue.mpg.de (download.is.tue.mpg.de)|192.124.27.139|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: download.php?domain=flame&sfile=FLAME2023.zip&resume=1 [following]
--2026-03-11 23:44:51--  https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2023.zip&resume=1
Reusing existing connection to download.is.tue.mpg.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 102652570 (98M) [application/octet-stream]
Saving to: ‘./FLAME2023.zip’

     0K .......... .......... .......... .......... ..........  0%  310K 5m23s
    50K .......... .......... .......... .......... ..........  0%  312K 5m22s
   100K .......... .......... .......... .......... ..........  0% 45.6M 3m35s
   150K .......... .......... .......... .......... ...

In [8]:
# Patch Pixel3DMM for torch>=2.6 checkpoint loading + facer indexing
from pathlib import Path

base = Path('/content/pixel3dmm/scripts')

# 1) torch>=2.6 changed torch.load default to weights_only=True
p = base / 'network_inference.py'
s = p.read_text()
old = 'load_from_checkpoint(model_checkpoint, strict=False)'
new = 'load_from_checkpoint(model_checkpoint, strict=False, weights_only=False)'
if old in s:
    p.write_text(s.replace(old, new))
    print('Patched:', p, '(weights_only=False)')
else:
    print('Already patched or pattern not found:', p)


Already patched or pattern not found: /content/pixel3dmm/scripts/network_inference.py
Already patched or pattern not found: /content/pixel3dmm/scripts/run_facer_segmentation.py


In [11]:
from pathlib import Path

# Patch cap4d/datasets/utils.py to handle missing .mp4 extension
p = Path("/content/cap4d") / "cap4d" / "datasets" / "utils.py"
content = p.read_text()

old_code = """def load_frame(
    video_path: Path,  # path to .mp4 or dir containing frames
    frame_id: np.ndarray,
):
    if (video_path).is_dir():"""

new_code = """def load_frame(
    video_path: Path,  # path to .mp4 or dir containing frames
    frame_id: np.ndarray,
):
    # PATCH: Check for .mp4 if path doesn't exist
    if not video_path.exists() and video_path.with_suffix('.mp4').exists():
        video_path = video_path.with_suffix('.mp4')

    if (video_path).is_dir():"""

if old_code in content:
    new_content = content.replace(old_code, new_code)
    p.write_text(new_content)
    print("Patched cap4d/datasets/utils.py successfully.")
else:
    print("Patch pattern not found. File might vary or already be patched.")
    # Check if already patched
    if "if not video_path.exists()" in content:
        print("File appears to be already patched.")


Patched cap4d/datasets/utils.py successfully.


In [14]:
import os

file_path = str(Path("/content/cap4d") / "scripts" / "pixel3dmm" / "l2cs_eye_tracker.py")

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
patched = False
for line in lines:
    # Locate the line with the problematic argument and comment it out
    if "face_detector_kwargs=" in line and not line.strip().startswith("#"):
        print(f"Patching line: {line.strip()}")
        new_lines.append(f"            # {line.strip()}  # Patched: argument removed\n")
        patched = True
    else:
        new_lines.append(line)

if patched:
    with open(file_path, "w") as f:
        f.writelines(new_lines)
    print("Successfully patched l2cs_eye_tracker.py")
    print("You can now rerun the tracking step.")
else:
    print("Pattern not found. The file might already be patched or the code structure is different.")


Patching line: face_detector_kwargs={"filter_threshold": 0.9}
Successfully patched l2cs_eye_tracker.py
You can now rerun the tracking step.


In [ ]:
        # for batch_idx in range(ceil(len(image_stack)/batch_size)):
        #     image_batch = torch.cat(image_stack[batch_idx*batch_size:(batch_idx+1)*batch_size], dim=0)
        #     frame_idx_batch = frame_stack[batch_idx*batch_size:(batch_idx+1)*batch_size]
        #     og_shape_batch = original_shapes[batch_idx*batch_size:(batch_idx+1)*batch_size]

        #     #if True:
        #     try:
        #         with torch.inference_mode():
        #             faces = face_detector(image_batch)
        #             torch.cuda.empty_cache()
        #             faces = face_parser(image_batch, faces, bbox_scale_factor=1.25)
        #             torch.cuda.empty_cache()

        #         seg_logits = faces['seg']['logits']
        #         back_ground = torch.all(seg_logits == 0, dim=1, keepdim=True).detach().squeeze(1).cpu().numpy()
        #         seg_probs = seg_logits.softmax(dim=1)  # nfaces x nclasses x h x w
        #         seg_classes = seg_probs.argmax(dim=1).detach().cpu().numpy().astype(np.uint8)
        #         seg_classes[back_ground] = seg_probs.shape[1] + 1

        #         for _iidx in range(seg_probs.shape[0]):
        #             idx = int(_iidx)
        #             if idx < 0 or idx >= len(frame_idx_batch):
        #                 continue
        #             frame = frame_idx_batch[idx]
        #             iidx = faces['image_ids'][_iidx].item()
        #             try:
        #                 I_color = viz_results(
        #                     image_batch[iidx:iidx+1],
        #                     seq_classes=seg_classes[_iidx:_iidx+1],
        #                     n_classes=seg_probs.shape[1] + 1,
        #                     suppress_plot=True
        #                 )
        #                 I_color.save(f'{out_seg_annot}/color_{frame}.png')
        #             except Exception as ex:
        #                 pass
        #             I = Image.fromarray(seg_classes[_iidx])
        #             I.save(f'{out_seg}/{frame}.png')
        #         torch.cuda.empty_cache()
        #     except Exception as exx:
        #         traceback.print_exc()
        #         continue

In [9]:
# # Optional upload: use this cell if INPUT_VIDEO_PATH is not already available in /content
# from google.colab import files
# uploaded = files.upload()
# if uploaded:
#     INPUT_VIDEO_PATH = f"/content/{next(iter(uploaded.keys()))}"
# print('INPUT_VIDEO_PATH =', INPUT_VIDEO_PATH)



INPUT_VIDEO_PATH = "/content/IMG_9809.mp4"

In [10]:
import os
import re
import shlex
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path
LOG_DIR = "/content/cap4d_logs"
os.makedirs(LOG_DIR, exist_ok=True)

# Rough ranges to give a practical expectation. These are not strict predictions.
ETA_HINTS_MIN = {
    "setup_clone": 1,
    "install_core": 8,
    "download_weights": 3,
    "install_pixel3dmm": 10,
    "tracking": 20,
    "generate_images": 30,
    "train_avatar": 45,
    "export_static": 1,
}
ETA_HINTS_MAX = {
    "setup_clone": 3,
    "install_core": 25,
    "download_weights": 10,
    "install_pixel3dmm": 35,
    "tracking": 90,
    "generate_images": 180,
    "train_avatar": 360,
    "export_static": 5,
}


def _now():
    return datetime.now().strftime("%H:%M:%S")


def run_logged(name, cmd, cwd=None, env=None, shell=False):
    """Run command with streamed logs, heartbeat, logfile, and rough ETA range."""
    log_path = Path(LOG_DIR) / f"{name}.log"
    start = time.time()
    last_line_time = [start]

    eta_min = ETA_HINTS_MIN.get(name)
    eta_max = ETA_HINTS_MAX.get(name)
    if eta_min is not None and eta_max is not None:
        print(f"[{_now()}] [{name}] ETA (rough): {eta_min}-{eta_max} min")

    print(f"[{_now()}] [{name}] START")
    print(f"[{_now()}] [{name}] CMD: {cmd if isinstance(cmd, str) else ' '.join(shlex.quote(c) for c in cmd)}")
    print(f"[{_now()}] [{name}] LOG: {log_path}")

    if shell:
        popen_cmd = cmd
    else:
        popen_cmd = cmd if isinstance(cmd, list) else shlex.split(cmd)

    process = subprocess.Popen(
        popen_cmd,
        cwd=cwd,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    stop_flag = {"stop": False}

    def heartbeat():
        while not stop_flag["stop"]:
            time.sleep(30)
            if stop_flag["stop"]:
                break
            elapsed = (time.time() - start) / 60.0
            silent = time.time() - last_line_time[0]
            msg = f"[{_now()}] [{name}] heartbeat: elapsed={elapsed:.1f}m"
            if silent >= 30:
                msg += f", no new output for {silent:.0f}s"
            if eta_max is not None:
                remaining_floor = max(0.0, eta_max - elapsed)
                msg += f", est remaining <= {remaining_floor:.1f}m"
            print(msg)

    t = threading.Thread(target=heartbeat, daemon=True)
    t.start()

    with log_path.open("w", encoding="utf-8") as f:
        for raw in process.stdout:
            line = raw.rstrip("\n")
            last_line_time[0] = time.time()
            elapsed = time.time() - start
            prefix = f"[{_now()}] [{name}] [+{elapsed:7.1f}s]"
            print(f"{prefix} {line}")
            f.write(raw)

    rc = process.wait()
    stop_flag["stop"] = True
    total = (time.time() - start) / 60.0

    if rc != 0:
        print(f"[{_now()}] [{name}] FAIL rc={rc} after {total:.1f}m")
        print(f"[{_now()}] [{name}] See log: {log_path}")
        raise RuntimeError(f"Step '{name}' failed (rc={rc}). Log: {log_path}")

    print(f"[{_now()}] [{name}] DONE in {total:.1f}m")
    return str(log_path)


print("Logging helper loaded. Logs will be written under:", LOG_DIR)


Logging helper loaded. Logs will be written under: /content/cap4d_logs


In [15]:
# Run full static pipeline in one command with logs
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

run_logged(
    name="tracking",
    cmd=[
        "bash", "scripts/track_video_pixel3dmm.sh",
        INPUT_VIDEO_PATH,
        f"{OUTPUT_PATH}/reference_tracking",
        "--max_n_ref", str(MAX_N_REF),
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="generate_images",
    cmd=[
        "python", "cap4d/inference/generate_images.py",
        "--config_path", "configs/generation/low_quality.yaml" if QUALITY == "balanced" else ("configs/generation/high_quality.yaml" if QUALITY == "max" else "configs/generation/debug.yaml"),
        "--reference_data_path", f"{OUTPUT_PATH}/reference_tracking",
        "--output_path", f"{OUTPUT_PATH}/mmdm",
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="train_avatar",
    cmd=[
        "python", "gaussianavatars/train.py",
        "--config_path", "configs/avatar/low_quality.yaml" if QUALITY == "balanced" else ("configs/avatar/high_quality.yaml" if QUALITY == "max" else "configs/avatar/debug.yaml"),
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="export_static",
    cmd=[
        "python", "gaussianavatars/export_static_ply.py",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--output_ply", f"{OUTPUT_PATH}/raw_static.ply",
        "--timestep", str(TIMESTEP),
    ],
    cwd="/content/cap4d",
)


[23:59:11] [tracking] ETA (rough): 20-90 min
[23:59:11] [tracking] START
[23:59:11] [tracking] CMD: bash scripts/track_video_pixel3dmm.sh /content/IMG_9809.mp4 /content/cap4d/examples/output/custom_static/reference_tracking --max_n_ref 100
[23:59:11] [tracking] LOG: /content/cap4d_logs/tracking.log
[23:59:11] [tracking] [+    0.0s] Processing video:
[23:59:11] [tracking] [+    0.0s]   Input:  /content/IMG_9809.mp4
[23:59:11] [tracking] [+    0.0s]   Output: /content/cap4d/examples/output/custom_static/reference_tracking
[23:59:11] [tracking] [+    0.0s] Found input video ('/content/IMG_9809.mp4').
[23:59:11] [tracking] [+    0.0s] Running Pixel3DMM script
[23:59:24] [tracking] [+   13.3s] /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
[23:59:24] [tracking] [+   13.3s]   warnings.warn(
[23:59:24] [tracking] [+   13.3s] /usr/local/lib


KeyboardInterrupt



In [ ]:
from google.colab import files
ply_path = f"{OUTPUT_PATH}/raw_static.ply"
print('Exported PLY:', ply_path)
files.download(ply_path)